# Transformers

## Learning Objectives
1. Build a Transformer block from scratch in numpy: attention, FFN, residual, LayerNorm
2. Train a PyTorch `nn.TransformerEncoderLayer` classifier; handle OOM on long sequences
3. Fine-tune a HuggingFace AutoModel for text classification
4. Compare sinusoidal vs learned vs no positional encoding; Transformer vs LSTM on long seqs


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import time

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## Level 1 — Transformer Block from Scratch (numpy)

In [ ]:
# ── Level 1: self-attention + FFN + residual + LayerNorm (numpy) ──────────
import numpy as np

np.random.seed(42)

def softmax(x, axis=-1):
    e = np.exp(x - x.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)

def layer_norm(x, eps=1e-6):
    mean = x.mean(axis=-1, keepdims=True)
    std  = x.std(axis=-1, keepdims=True)
    return (x - mean) / (std + eps)

def self_attention(Q, K, V, d_k):
    # Scaled dot-product attention. Q, K, V: (seq, d_k)
    scores = Q @ K.T / np.sqrt(d_k)   # (seq, seq)
    attn   = softmax(scores)            # (seq, seq)
    return attn @ V, attn              # (seq, d_v), (seq, seq)

def ffn(x, W1, b1, W2, b2):
    # Two-layer feed-forward with GELU approx.
    h = np.maximum(0, x @ W1 + b1)    # ReLU (GELU approx for demo)
    return h @ W2 + b2

# Dimensions
BATCH, SEQ, D_MODEL, D_K, D_FF = 1, 6, 16, 8, 32

# Random input and weights
X = np.random.randn(SEQ, D_MODEL)
Wq = np.random.randn(D_MODEL, D_K) * 0.1
Wk = np.random.randn(D_MODEL, D_K) * 0.1
Wv = np.random.randn(D_MODEL, D_K) * 0.1
Wo = np.random.randn(D_K, D_MODEL)  * 0.1
W1 = np.random.randn(D_MODEL, D_FF) * 0.1
b1 = np.zeros(D_FF)
W2 = np.random.randn(D_FF, D_MODEL) * 0.1
b2 = np.zeros(D_MODEL)

print(f"Input X:      {X.shape}")
Q = X @ Wq;  K = X @ Wk;  V = X @ Wv
print(f"Q/K/V:        {Q.shape}  (each)")

attn_out, attn_wts = self_attention(Q, K, V, D_K)
print(f"Attention out: {attn_out.shape}")

proj = attn_out @ Wo
x1   = layer_norm(X + proj)   # residual + LayerNorm
print(f"After LN1:    {x1.shape}")

ff_out = ffn(x1, W1, b1, W2, b2)
x2     = layer_norm(x1 + ff_out)   # residual + LayerNorm
print(f"After FFN+LN2:{x2.shape}  (same as input — key property!)")
print(f"\nAttention weights (softmax): min={attn_wts.min():.4f}  max={attn_wts.max():.4f}  sum_per_row≈{attn_wts.sum(axis=-1)}")


## Level 2 — PyTorch TransformerEncoderLayer for Classification

In [ ]:
# ── Level 2: nn.TransformerEncoderLayer sequence classifier ─────────────
import torch, time
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)

D_MODEL, N_HEADS, DIM_FF, N_LAYERS = 64, 4, 256, 3

class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size=200, d_model=D_MODEL, n_heads=N_HEADS,
                 dim_ff=DIM_FF, n_layers=N_LAYERS, n_class=2, max_len=512):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d_model)
        self.pos = nn.Embedding(max_len, d_model)     # learned positional encoding
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=dim_ff,
            dropout=0.1, batch_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.fc      = nn.Linear(d_model, n_class)

    def forward(self, x):
        positions = torch.arange(x.size(1), device=x.device).unsqueeze(0)
        h = self.emb(x) + self.pos(positions)
        h = self.encoder(h)
        return self.fc(h.mean(dim=1))   # mean pooling

model = TransformerClassifier().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Transformer parameters: {total_params:,}")

# Synthetic data: classify by majority token value
SEQ_LEN = 50
X_all = torch.randint(1, 200, (800, SEQ_LEN))
y_all = (X_all.float().mean(dim=1) > 100).long()
loader = DataLoader(TensorDataset(X_all[:700], y_all[:700]), batch_size=32, shuffle=True)
X_v, y_v = X_all[700:].to(device), y_all[700:].to(device)

opt = torch.optim.Adam(model.parameters(), lr=1e-4)
for epoch in range(15):
    model.train()
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        try:
            opt.zero_grad()
            loss = F.cross_entropy(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                print("OOM: reduce batch_size or sequence length"); break
            raise

model.eval()
with torch.no_grad():
    acc = (model(X_v).argmax(1) == y_v).float().mean().item()
print(f"\nTransformer val accuracy: {acc:.3f}")


## Real-World Example 1 — HuggingFace AutoModel Text Classification

In [ ]:
# ── RW1: HuggingFace AutoModel (simulated to avoid download) ─────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

# NOTE: This example shows the exact HuggingFace fine-tuning pattern.
# In production, replace SimulatedBERT with:
#   from transformers import AutoModel
#   backbone = AutoModel.from_pretrained("bert-base-uncased")

torch.manual_seed(42)
HIDDEN, N_LAYERS, N_CLASS = 256, 4, 3

class SimulatedBERT(nn.Module):
    """Structural substitute for AutoModel — same interface, no download."""
    def __init__(self):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=HIDDEN, nhead=4, dim_feedforward=HIDDEN * 4,
            dropout=0.1, batch_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=N_LAYERS)

    def forward(self, input_ids, attention_mask=None):
        # Simulated: treat input_ids as float embeddings
        h = self.encoder(input_ids)
        # Return dict matching HuggingFace BaseModelOutput
        return {"last_hidden_state": h}

class SequenceClassifier(nn.Module):
    """Wraps backbone — mirrors AutoModelForSequenceClassification structure."""
    def __init__(self, backbone, n_class=N_CLASS):
        super().__init__()
        self.backbone = backbone
        self.dropout  = nn.Dropout(0.1)
        self.fc       = nn.Linear(HIDDEN, n_class)

    def forward(self, input_ids, attention_mask=None):
        outputs  = self.backbone(input_ids, attention_mask)
        cls_repr = outputs["last_hidden_state"][:, 0]   # CLS token
        return self.fc(self.dropout(cls_repr))

backbone  = SimulatedBERT()
clf_model = SequenceClassifier(backbone).to(device)

# Synthetic dataset
SEQ = 30
X   = torch.randn(600, SEQ, HIDDEN)
y   = torch.randint(0, N_CLASS, (600,))
loader = DataLoader(TensorDataset(X[:500], y[:500]), batch_size=16, shuffle=True)
X_v, y_v = X[500:].to(device), y[500:].to(device)

# Standard HuggingFace fine-tuning: AdamW + linear LR warmup
opt = torch.optim.AdamW(clf_model.parameters(), lr=2e-5, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.LinearLR(opt, start_factor=0.1, total_iters=5)

for epoch in range(10):
    clf_model.train()
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        F.cross_entropy(clf_model(xb), yb).backward()
        torch.nn.utils.clip_grad_norm_(clf_model.parameters(), 1.0)
        opt.step()
    if epoch < 5: scheduler.step()

clf_model.eval()
with torch.no_grad():
    acc = (clf_model(X_v).argmax(1) == y_v).float().mean().item()
print(f"SimulatedBERT fine-tuning val_acc: {acc:.3f}")
print("In production: replace SimulatedBERT with AutoModel.from_pretrained(...)")


## Real-World Example 2 — Positional Encoding Ablation

In [ ]:
# ── RW2: sinusoidal vs learned vs no positional encoding ─────────────────
import torch, math, copy
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
D, SEQ, N_CLASS = 64, 40, 2

def sinusoidal_pe(max_len, d_model):
    # Classic Vaswani et al. positional encoding.
    pe  = torch.zeros(max_len, d_model)
    pos = torch.arange(0, max_len).unsqueeze(1).float()
    div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div)
    return pe   # (max_len, d_model)

class TransformerWithPE(nn.Module):
    def __init__(self, pe_type="sinusoidal"):
        super().__init__()
        self.proj    = nn.Linear(1, D)
        self.pe_type = pe_type
        if pe_type == "learned":
            self.pos = nn.Embedding(SEQ, D)
        elif pe_type == "sinusoidal":
            pe = sinusoidal_pe(SEQ, D)
            self.register_buffer("pe_buf", pe)
        enc = nn.TransformerEncoderLayer(D, nhead=4, dim_feedforward=128,
                                          dropout=0.0, batch_first=True)
        self.enc = nn.TransformerEncoder(enc, num_layers=2)
        self.fc  = nn.Linear(D, N_CLASS)

    def forward(self, x):           # x: (B, T)
        h = self.proj(x.unsqueeze(-1))   # (B, T, D)
        if self.pe_type == "learned":
            pos = torch.arange(x.size(1), device=x.device).unsqueeze(0)
            h   = h + self.pos(pos)
        elif self.pe_type == "sinusoidal":
            h = h + self.pe_buf[:x.size(1)].unsqueeze(0)
        return self.fc(self.enc(h).mean(1))

# Task: first token > last token
X    = torch.randn(800, SEQ)
y    = (X[:, 0] > X[:, -1]).long()
loader = DataLoader(TensorDataset(X[:700], y[:700]), batch_size=32, shuffle=True)
Xv, yv = X[700:].to(device), y[700:].to(device)

results = {}
for pe_type in ["none", "sinusoidal", "learned"]:
    m   = TransformerWithPE(pe_type).to(device)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    for _ in range(20):
        m.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            F.cross_entropy(m(xb), yb).backward()
            opt.step()
    m.eval()
    with torch.no_grad():
        acc = (m(Xv).argmax(1) == yv).float().mean().item()
    results[pe_type] = acc
    print(f"PE type={pe_type:<12}  val_acc={acc:.3f}")

best = max(results, key=results.get)
print(f"\nBest PE: {best}  (learned and sinusoidal outperform no-PE on position-sensitive tasks)")


## Real-World Example 3 — Transformer vs LSTM on Long Sequences

In [ ]:
# ── RW3: Transformer vs LSTM — accuracy at sequence lengths 64→512 ────────
import torch, time
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
VOCAB, EMBED, HIDDEN = 100, 32, 64

class SimpleLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb  = nn.Embedding(VOCAB, EMBED)
        self.lstm = nn.LSTM(EMBED, HIDDEN, batch_first=True)
        self.fc   = nn.Linear(HIDDEN, 2)
    def forward(self, x):
        _, (h, _) = self.lstm(self.emb(x))
        return self.fc(h.squeeze(0))

class SimpleTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(VOCAB, EMBED)
        enc = nn.TransformerEncoderLayer(EMBED, nhead=4, dim_feedforward=128,
                                          batch_first=True, dropout=0.0)
        self.enc = nn.TransformerEncoder(enc, num_layers=2)
        self.fc  = nn.Linear(EMBED, 2)
    def forward(self, x):
        return self.fc(self.enc(self.emb(x)).mean(1))

def run(ModelCls, seq_len, n=500, epochs=15):
    ids   = torch.randint(1, VOCAB, (n, seq_len))
    label = (ids[:, -1] > VOCAB // 2).long()   # last-token task
    ld    = DataLoader(TensorDataset(ids[:400], label[:400]), batch_size=32, shuffle=True)
    Xv, yv = ids[400:].to(device), label[400:].to(device)
    m   = ModelCls().to(device)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    t0  = time.perf_counter()
    for _ in range(epochs):
        m.train()
        for xb, yb in ld:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            F.cross_entropy(m(xb), yb).backward()
            opt.step()
    elapsed = time.perf_counter() - t0
    m.eval()
    with torch.no_grad():
        acc = (m(Xv).argmax(1) == yv).float().mean().item()
    return acc, elapsed

print(f"{'Seq Len':>8}  {'LSTM Acc':>9}  {'LSTM s':>7}  {'Transf Acc':>11}  {'Transf s':>9}")
print("-" * 55)
for sl in [64, 128, 256, 512]:
    acc_l, t_l = run(SimpleLSTM,         sl)
    acc_t, t_t = run(SimpleTransformer,  sl)
    print(f"{sl:>8}  {acc_l:>9.3f}  {t_l:>7.2f}  {acc_t:>11.3f}  {t_t:>9.2f}")
print("\nTransformer scales better to long sequences; LSTM degrades due to vanishing gradients.")
